# 04 · Core test (H0)

**Question:** after a made or missed shot, does an NBA player's next shot change in distance, zone or
difficulty-adjusted make probability, compared with that same player's own baseline?

Runs once on the **analysis half** (D-040). Specifications: D-036 to D-039. The interpretation in
plain units is Massimo's and is written after this notebook, not in it.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hothand import core
from hothand.config import FIGURES, PROCESSED, load_config
from hothand.features import FEATURES, SHOTS
from hothand.pairs import ZONE_NAMES

pd.set_option("display.max_columns", 40, "display.width", 200)
cfg = load_config()
print("Python:", sys.executable)
print("pandas:", pd.__version__)
HALF = "analysis"
d = core.prepare(pd.read_parquet(FEATURES), cfg, HALF)
shots = pd.read_parquet(SHOTS)
shots = shots[shots.half == HALF]
print(f"{len(d):,} pairs, {d.player_id.nunique():,} players, half = {HALF}")

## 1. Look before modelling (playbook 5.1)

Raw distributions only: next-shot distance and zone mix after a make vs after a miss, and each player's
raw paired difference. These are not yet adjusted for where the previous shot was taken.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
bins = np.arange(0, 32, 1)
for made, label, color in [(1, "after a make", "#1f77b4"), (0, "after a miss", "#d95f02")]:
    axes[0].hist(d.loc[d.prev_made == made, "coord_distance_ft"], bins=bins, density=True,
                 histtype="step", lw=1.6, color=color, label=label)
axes[0].set_xlabel("Next-shot distance (ft)"); axes[0].set_ylabel("Share of shots")
axes[0].legend(frameon=False)
mix = d.groupby("prev_made").zone.value_counts(normalize=True).unstack()[ZONE_NAMES].mul(100)
x = np.arange(len(ZONE_NAMES))
axes[1].bar(x - 0.2, mix.loc[1], 0.4, label="after a make", color="#1f77b4")
axes[1].bar(x + 0.2, mix.loc[0], 0.4, label="after a miss", color="#d95f02")
axes[1].set_xticks(x, ["Rim", "Paint", "Mid-range", "Corner 3", "Above-\nbreak 3"])
axes[1].set_ylabel("% of next shots"); axes[1].legend(frameon=False)
fig.suptitle(f"Next shot after a make vs a miss, raw (analysis half, {len(d):,} pairs)", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "stage5_raw_next_shot.png", dpi=150)
mix.round(1)

In [ ]:
per_player = (d.groupby(["player_id", "prev_made"]).distance_vs_baseline_ft.mean().unstack()
              .dropna())
per_player["diff_make_minus_miss"] = per_player[1] - per_player[0]
counts = d.groupby("player_id").prev_made.agg(["sum", "size"])
per_player = per_player.join(counts)
big = per_player[(per_player["sum"] >= 50) & (per_player["size"] - per_player["sum"] >= 50)]
print(f"Players with 50+ pairs after makes and after misses: {len(big)}")
big.diff_make_minus_miss.describe().round(2)

## 2. Models 0-3 (playbook 5.3)

Each model adds one block: 0 = previous shot made only; 1 = + previous shot's distance, zone and xFG
(mechanical correlation); 2 = + game context; 3 = + player-season fixed effects. Standard errors
clustered by player.

In [ ]:
outcomes = ["distance_vs_baseline_ft"] + core.ZONE_OUTCOMES + ["next_surprise"]
models = pd.concat([core.fit_models(d, o) for o in outcomes], ignore_index=True)
surprise = core.fit_models(d, "distance_vs_baseline_ft", "prev_surprise")
models = pd.concat([models, surprise], ignore_index=True)
models.pivot_table(index=["outcome", "treatment"], columns="model", values="estimate", sort=False).round(4)

In [ ]:
models.loc[models.model == "Model 3", ["outcome", "treatment", "estimate", "ci_low", "ci_high",
                                       "p_value", "n_pairs", "n_players"]].round(4)

## 3. Permutation test (playbook 5.4, D-039)

Shot order is shuffled within each player-game (and, as a check, within each player-game-period),
1,000 times. Bias-corrected effect = observed difference minus the mean shuffled difference.

In [ ]:
n_perm = cfg["inference"]["n_permutations"]
seed = cfg["random_seed"]
perm_game, null_game = core.permutation_test(shots, d, n_perm, seed, ["game_id", "player_id"])
perm_period, null_period = core.permutation_test(shots, d, n_perm, seed,
                                                 ["game_id", "player_id", "period"])
perm = pd.concat([perm_game, perm_period], ignore_index=True)
perm.round(5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
labels = {"next_surprise": "Make prob. above xFG (pp)", "distance_vs_baseline_ft": "Distance (ft)",
          "rel_three": "Three-point share (pp)"}
scale = {"next_surprise": 100, "distance_vs_baseline_ft": 1, "rel_three": 100}
for ax, k in zip(axes, labels):
    ax.hist(null_game[k] * scale[k], bins=40, color="#bbb")
    obs = perm_game.set_index("outcome").loc[k, "observed"] * scale[k]
    ax.axvline(obs, color="#d95f02", lw=2, label="observed")
    ax.set_title(labels[k], fontsize=10); ax.set_yticks([])
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle("After a make minus after a miss: observed vs 1,000 within-player-game shuffles", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "stage5_permutation_null.png", dpi=150)

## 4. Second method: mixed-effects model (playbook 5.5)

In [ ]:
mixed = pd.DataFrame([core.fit_mixed(d, o) for o in ["distance_vs_baseline_ft", "rel_three"]])
m3 = models[(models.model == "Model 3") & (models.treatment == "prev_made")].set_index("outcome")
compare = mixed.set_index("outcome")[["estimate", "ci_low", "ci_high", "converged"]].join(
    m3[["estimate", "ci_low", "ci_high"]], rsuffix="_model3")
compare["same_sign"] = np.sign(compare.estimate) == np.sign(compare.estimate_model3)
compare["ratio_mixed_to_model3"] = compare.estimate / compare.estimate_model3
compare.round(4)

## 5. Save the results table

In [ ]:
models["method"] = "regression"
mixed["method"] = "mixed"
mixed["treatment"] = "prev_made"
perm_out = perm.rename(columns={"bias_corrected": "estimate"}).assign(
    method="permutation", treatment="prev_made", model=lambda x: "shuffle within " + x.within)
results = pd.concat([models, mixed, perm_out], ignore_index=True)
results.insert(0, "half", HALF)
results.to_csv(PROCESSED / "results_core.csv", index=False)
print(f"{len(results)} rows -> data/processed/results_core.csv")